# SEEG Manifold Analysis - Quick Start Guide

This notebook demonstrates the basic workflow for analyzing SEEG data:

1. Loading data
2. Preprocessing
3. Dimensionality estimation
4. Multi-method dimensionality reduction
5. Visualization

## Setup

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Project imports
import sys
sys.path.insert(0, '..')

from src.io import load_seeg_data, SEEGData
from src.preprocessing import preprocess_pipeline
from src.manifold import estimate_dimensionality, compare_reductions

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data

### Option A: Load from MATLAB file

In [ ]:
# Uncomment and modify path to load your data
# data = load_seeg_data('path/to/your/data.mat')
# print(data)

### Option B: Generate synthetic data for demonstration

In [ ]:
# Generate synthetic SEEG-like data for demonstration
np.random.seed(42)

# Parameters
n_channels = 64
sfreq = 1000  # Hz
duration = 60  # seconds
n_timepoints = int(sfreq * duration)

# Generate data with some structure:
# - Low-dimensional oscillatory component
# - High-dimensional noise

t = np.arange(n_timepoints) / sfreq

# Create a few latent oscillatory sources
n_sources = 5
sources = np.zeros((n_sources, n_timepoints))
sources[0] = np.sin(2 * np.pi * 10 * t)  # 10 Hz (alpha)
sources[1] = np.sin(2 * np.pi * 6 * t)   # 6 Hz (theta)
sources[2] = np.sin(2 * np.pi * 25 * t)  # 25 Hz (beta)
sources[3] = np.sin(2 * np.pi * 10 * t + np.pi/4)  # Phase-shifted alpha
sources[4] = 0.5 * np.sin(2 * np.pi * 40 * t)  # 40 Hz (gamma)

# Random mixing matrix (simulating volume conduction)
mixing = np.random.randn(n_channels, n_sources)

# Mix sources and add noise
signal = mixing @ sources
noise = 0.5 * np.random.randn(n_channels, n_timepoints)
synthetic_data = signal + noise

# Create SEEGData object
ch_names = [f'sEEG{i+1}' for i in range(n_channels)]
data = SEEGData(
    data=synthetic_data,
    sfreq=sfreq,
    ch_names=ch_names
)

print(f"Created synthetic data: {data}")
print(f"True latent dimensionality: {n_sources}")

### Visualize raw data

In [ ]:
# Plot a few channels
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

plot_duration = 2  # seconds
plot_samples = int(plot_duration * sfreq)

for i, ax in enumerate(axes):
    ax.plot(t[:plot_samples], data.data[i, :plot_samples], 'b-', linewidth=0.5)
    ax.set_ylabel(data.ch_names[i])
    ax.set_xlim([0, plot_duration])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw SEEG Data (first 4 channels)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
# Apply preprocessing pipeline
processed, preproc_info = preprocess_pipeline(
    data.data,
    sfreq=data.sfreq,
    lowcut=1,           # High-pass at 1 Hz
    highcut=100,        # Low-pass at 100 Hz
    notch_freq=50,      # Remove 50 Hz line noise
    epoch_length=2.0,   # 2-second epochs
    epoch_overlap=0.5,  # 50% overlap
    verbose=True
)

print(f"\nPreprocessed data shape: {processed.shape}")
print(f"  - {processed.shape[0]} epochs")
print(f"  - {processed.shape[1]} channels")
print(f"  - {processed.shape[2]} timepoints per epoch")

## 3. Dimensionality Estimation

In [ ]:
# Estimate intrinsic dimensionality
dim_results = estimate_dimensionality(
    processed,
    methods=['pca_variance', 'pca_elbow', 'mle'],
    variance_threshold=0.95,
    verbose=True
)

In [ ]:
# Visualize PCA explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
n_show = min(30, len(dim_results['pca_variance_ratio']))
axes[0].bar(range(1, n_show+1), dim_results['pca_variance_ratio'][:n_show])
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].axvline(dim_results['pca_elbow'], color='r', linestyle='--', label=f"Elbow: {dim_results['pca_elbow']}")
axes[0].legend()

# Cumulative variance
cumvar = np.cumsum(dim_results['pca_variance_ratio'])
axes[1].plot(range(1, len(cumvar)+1), cumvar, 'b-')
axes[1].axhline(0.95, color='r', linestyle='--', label='95% threshold')
axes[1].axvline(dim_results['pca_variance'], color='g', linestyle='--', 
                label=f"95% at {dim_results['pca_variance']} dims")
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlim([0, n_show])
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nConsensus dimensionality estimate: {dim_results['consensus']}")
print(f"(True latent dimensionality was: {n_sources})")

## 4. Multi-Method Dimensionality Reduction

In [ ]:
# Compare multiple reduction methods
reduction_results = compare_reductions(
    processed,
    methods=['pca', 'umap', 'tsne', 'isomap'],
    n_components=3,
    compute_metrics=True,
    verbose=True
)

In [ ]:
# Visualize embeddings
fig = plt.figure(figsize=(16, 12))

methods = list(reduction_results.keys())
n_methods = len(methods)

for i, method in enumerate(methods):
    ax = fig.add_subplot(2, 2, i+1, projection='3d')
    
    embedding = reduction_results[method].embedding
    
    # Color by time (epoch index)
    colors = np.arange(embedding.shape[0])
    
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        embedding[:, 2],
        c=colors,
        cmap='viridis',
        s=20,
        alpha=0.7
    )
    
    corr = reduction_results[method].metrics.get('distance_correlation', 'N/A')
    corr_str = f"{corr:.3f}" if isinstance(corr, float) else corr
    
    ax.set_title(f"{method.upper()}\n(dist. corr: {corr_str})")
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')
    ax.set_zlabel('Dim 3')

plt.suptitle('3D Embeddings Comparison (colored by time)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Next Steps

From here, you can:

1. **Analyze the manifold structure**: Look for patterns, clusters, or trajectories
2. **Apply topological data analysis**: Use persistent homology to find robust features
3. **Search for symmetries**: Test for rotation invariance or other group structures
4. **Compare with experimental conditions**: If you have task data, compare resting vs. task manifolds

See the other notebooks in this folder for more advanced analyses.

---

## Appendix: Loading Your Own Data from MATLAB

To use your own data, export it from MATLAB using:

```matlab
% In MATLAB:
seeg_data = struct();
seeg_data.data = your_data;      % (n_channels x n_timepoints)
seeg_data.sfreq = 1000;          % Your sampling rate
seeg_data.ch_names = ch_names;   % Cell array of channel names

save('my_seeg_data.mat', 'seeg_data', '-v7.3');
```

Then load in Python:

```python
data = load_seeg_data('my_seeg_data.mat')
```